# Window Functions, Ranking & Quantile Binning (5+ Years Interview Guide)
Exhaustive senior guide to ranking protocols, equal-frequency quantile binning (qcut), interval discretization (cut), time-based rolling windows, and cumulative statistics.

### Key 5-Year Interview Concepts Covered:
- **Ranking Protocols**: `rank(method='dense')`, `'min'`, `'max'`, `'first'`, `'average'`, and percentile ranks.
- **Discretization & Binning**: Equal-frequency `pd.qcut()` (quantiles) vs equal-width `pd.cut()` (fixed intervals).
- **Time-Based Rolling Windows**: `df.rolling('7D')` over irregular datetime indices.
- **Lead / Lag Offsets**: Calculating multi-step shifts with `.shift()` and percentage changes with `.pct_change()`.
- **Cumulative Running Metrics**: `.cumsum()`, `.cummax()`, `.cummin()`, and `.cumprod()`.

This interactive notebook is fully customized using the Fintech dataset `data/raw_transactions.csv`.

In [ ]:
# Setup imports & load dataset
import pandas as pd
import numpy as np
import os

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path, na_values=['Nan', ''])
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(2)

## Section 1: Ranking Protocols & Percentiles

### Ranking Methods (`rank(method=...)`)
**Explanation**: The `.rank()` method computes numerical rank positions. The `method` parameter controls tie-breaking: `'average'` (assigns average rank to ties), `'min'` (lowest rank), `'max'` (highest rank), `'first'` (order of appearance), and `'dense'` (like min, but rank always increments by 1 without gaps). Passing `pct=True` computes percentile ranks (0.0 to 1.0).

**Syntax**: `df['amount'].rank(method='dense', ascending=False)` / `df['amount'].rank(pct=True)`

In [ ]:
sample = df[['transaction_id', 'region', 'transaction_amount']].dropna().head(6).copy()
sample['Rank_Dense'] = sample['transaction_amount'].rank(method='dense', ascending=False)
sample['Rank_Pct'] = sample['transaction_amount'].rank(pct=True)
print(sample)

### Group-Specific Rankings (`groupby.rank`)
**Explanation**: Combining `.groupby()` with `.rank()` computes rankings within specific partitions (e.g. ranking transactions within each geographic region). This is equivalent to SQL `RANK() OVER (PARTITION BY region ORDER BY amount DESC)`.

**Syntax**: `df.groupby('region')['amount'].rank(ascending=False, method='dense')`

In [ ]:
df['regional_rank'] = df.groupby('region')['transaction_amount'].rank(ascending=False, method='dense')
print(df[['region', 'transaction_amount', 'regional_rank']].sort_values(['region', 'regional_rank']).head(6))

## Section 2: Discretization & Binning (`qcut` vs `cut`)

### Equal-Frequency Binning (`pd.qcut`)
**Explanation**: `pd.qcut(series, q=4)` divides data into quantiles such that each bin contains approximately the EXACT SAME number of observations (e.g. quartiles, deciles). This is standard for credit risk scoring and customer wealth tiering.

**Syntax**: `pd.qcut(df['amount'], q=4, labels=['Low', 'Med', 'High', 'VIP'])`

In [ ]:
amounts = df['transaction_amount'].dropna()
df['amount_quartile'] = pd.qcut(amounts, q=4, labels=['Q1_Low', 'Q2_Medium', 'Q3_High', 'Q4_VIP'])
print('Quartile Distribution:\n', df['amount_quartile'].value_counts())

### Equal-Width Interval Discretization (`pd.cut`)
**Explanation**: `pd.cut(series, bins=[0, 100, 500, 1500])` segments data into explicit numerical value ranges. Unlike `qcut`, the bins have equal mathematical widths (or custom specified thresholds), which may result in uneven sample counts per bin.

**Syntax**: `pd.cut(df['amount'], bins=[0, 100, 500, 1500], labels=['Micro', 'Standard', 'Major'])`

In [ ]:
custom_bins = [0, 200, 800, 2000]
custom_labels = ['Micro (<$200)', 'Standard ($200-$800)', 'Major ($800+)']
df['amount_bracket'] = pd.cut(df['transaction_amount'], bins=custom_bins, labels=custom_labels)
print('Custom Bracket Counts:\n', df['amount_bracket'].value_counts())

## Section 3: Time Windows & Cumulative Statistics

### Irregular Time-Based Rolling Windows (`.rolling('7D')`)
**Explanation**: When data has an irregular timestamp index (events do not occur at fixed intervals), passing a time offset string like `'7D'` to `.rolling('7D')` computes a 7-day rolling window based on real time rather than a fixed row count. The index MUST be a sorted `DatetimeIndex`.

**Syntax**: `df.sort_index().rolling('7D')['amount'].mean()`

In [ ]:
df['parsed_date'] = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
ts_df = df.dropna(subset=['parsed_date']).set_index('parsed_date').sort_index()
ts_df['7D_Rolling_Avg'] = ts_df['transaction_amount'].rolling('7D').mean()
print(ts_df[['transaction_amount', '7D_Rolling_Avg']].head(5))

### Cumulative Running Operations (`cumsum`, `cummax`, `cummin`)
**Explanation**: `.cumsum()` computes running cumulative totals; `.cummax()` / `.cummin()` compute the running high-water / low-water marks. These execute in O(N) linear time and are essential for tracking account balances and peak portfolio drawdowns.

**Syntax**: `df['amount'].cumsum()` / `df['amount'].cummax()`

In [ ]:
ts_df['Cumulative_Revenue'] = ts_df['transaction_amount'].cumsum()
ts_df['Peak_Amount_Seen'] = ts_df['transaction_amount'].cummax()
print(ts_df[['transaction_amount', 'Cumulative_Revenue', 'Peak_Amount_Seen']].head(4))

## Section: Senior Fintech Interview Questions (5+ Years Experience)

### Q1: Top 2 Highest Transactions Per Region via Dense Rank
**Explanation**: Filter the dataset to retrieve the top 2 highest transaction amounts for each geographical region using group ranking.

**Syntax**: `df[df.groupby('region')['transaction_amount'].rank(ascending=False, method='dense') <= 2]`

In [ ]:
top2_per_region = df[df.groupby('region')['transaction_amount'].rank(ascending=False, method='dense') <= 2]
print(top2_per_region[['region', 'transaction_id', 'transaction_amount']].sort_values(['region', 'transaction_amount'], ascending=[True, False]))